In [1]:
!pip install transformers bitsandbytes peft accelerate datasets PyMuPDF -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 76.7 MB/s eta 0:00:00


In [2]:
from datasets import Dataset
import shutil

In [3]:
import fitz

text_blocks = []
path = "/content/New Microsoft Word Document.docx"
with fitz.open(path) as pdf:
  for page in pdf:
    text = page.get_text("text")
    text_blocks.append(text)

In [4]:
import re

def split_paragraphs(pages):
  chunks = []
  for page_text in pages:
    question_answer_pairs = re.split(r"Q: ", page_text)
    for question_answer_pair in question_answer_pairs:
      if not question_answer_pair.strip():
        continue
      question = question_answer_pair.split("A: ")[0]
      answer = question_answer_pair.split("A: ")[1]
      combined = f"Question: {question.strip()}\nAnswer: {answer.strip()}"
      chunks.append(combined)

  return chunks

In [5]:
chunks = split_paragraphs(text_blocks)

In [6]:
data = [{'text': c} for c in chunks]

In [7]:
dataset = Dataset.from_list(data)

In [8]:
dataset

Dataset({
    features: ['text'],
    num_rows: 100
})

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

In [10]:
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [11]:
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [12]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [13]:
def tokenize_fn(examples):
  tokens = tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [14]:
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [15]:
tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})

In [16]:
non_instruction_model_path = "/content/non-instruction_model"

In [17]:
shutil.unpack_archive("/content/non-instruction_model.zip", non_instruction_model_path)

In [19]:
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
    )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [26]:
model = PeftModel.from_pretrained(base_model, non_instruction_model_path)
model.enable_input_require_grads()

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [23]:
training_args = TrainingArguments(
    output_dir="/content/trained_model",
    per_device_train_batch_size=2,
    num_train_epochs=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    report_to="none")

In [27]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized
)

In [28]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss


Step,Training Loss
50,5.762492
100,5.751799


TrainOutput(global_step=100, training_loss=5.757145385742188, metrics={'train_runtime': 170.6089, 'train_samples_per_second': 1.172, 'train_steps_per_second': 0.586, 'total_flos': 636296468889600.0, 'train_loss': 5.757145385742188, 'epoch': 2.0})

In [29]:
from google.colab import files

# shutil.make_archive(output_name, format, folder_to_zip)
shutil.make_archive('/content/instruction_model', 'zip', '/content/trained_model/checkpoint-100')

files.download('/content/instruction_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
prompt = "How to reach the non-target customers"

In [32]:
formatted_prompt = f"Question: {prompt}\nAnswer: "

In [33]:
input = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

In [34]:
outputs = model.generate(
    **input,
    max_new_tokens=200,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    repetition_penalty=1.1
)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


In [35]:
tokenizer.decode(outputs[0], skip_special_tokens=True)

'Question: How to reach the non-target customers\nAnswer: 1. Identify the target customer: The first step is to identify your target customer based on their demographics, interests, and behaviors. Once you have identified your target customer, create a list of potential email contacts who share similar characteristics.\n2. Segment your email list: Create separate lists for different segments of your target customer base, such as those who are currently subscribed to your email newsletter or those who recently made a purchase from your website.\n3. Use segment'